In [1]:
from satpy import Scene
import pandas as pd
import ast
import s3fs
from tqdm import tqdm

# Connect to AWS public buckets
fs = s3fs.S3FileSystem(anon=True)

In [53]:
def compile_tcprimed_files(years=[2023, 2024], basins=['AL', 'EP', 'IO', 'SH', 'WP']):
    """
    Checks AWS S3 for TC-Primed files and compiles a DataFrame with the file paths and metadata.
    """
    df = pd.DataFrame(columns=['year', 'basin', 'storm_number', 'storm_id', 'satellite', 'time', 'size', 'file_path'])

    for year in tqdm(years, desc="Processing years"):
        for basin in tqdm(basins, desc=f"Processing basins for year {year}"):
            # Construct the S3 path for the TC-Primed files
            s3_base_path = f's3://noaa-nesdis-tcprimed-pds/v01r01/final/{year}/{basin}'
            
            try:
                folders = fs.ls(s3_base_path)
                for folder in folders:
                    number = folder.split('/')[-1]  # Extract the storm number from the folder name
                    files = fs.ls(folder)
                    for file in files:
                        if file.endswith('.nc'):
                            # Extract metadata from the file name
                            file_name = file.split('/')[-1]
                            file_name = file_name.split('.')[0]  # Remove the .nc extension
                            parts = file_name.split('_') # TCPRIMED_v01r01-final_AL122023_AMSR2_GCOMW1_060020_20230830024032
                            if len(parts) == 7:
                                _, _, storm_name, satellite, _, _, time  = parts
                            elif len(parts) == 6:
                                _, _, storm_name, satellite, time, _  = parts
                                time = time[1:] # Remove the leading 's' from the time string

                            size = fs.info(file)['Size'] / (1024 ** 3) # Convert to GB
                                
                            df_tmp = pd.DataFrame({
                                'year': [year],
                                'basin': [basin],
                                'storm_number': [number],
                                'storm_id': [storm_name],
                                'satellite': [satellite],
                                'time': [time],
                                'size': [size],
                                'file_path': [file]
                            })
                            df = pd.concat([df, df_tmp], ignore_index=True)
            except Exception as e:
                print(f"Error accessing {s3_base_path}: {e}")
                continue

    return df
                


In [54]:
df = compile_tcprimed_files()

Processing years:   0%|          | 0/2 [00:00<?, ?it/s]/var/folders/k9/_hm_155s5md_7xqqlb895t980000gn/T/ipykernel_15949/3170595878.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_tmp], ignore_index=True)
Processing years: 100%|██████████| 2/2 [00:10<00:00,  5.14s/it]


In [55]:
df.to_csv('tcprimed-files-[2023-2024].csv', index=False)

In [2]:
df = pd.read_csv('tcprimed-files-[2023-2024].csv')

In [3]:
df['size'].sum()

np.float64(146.6068328479293)